# 📊 PPT Online Dataset Downloader
Downloads `.pptx` files from the PPT Online torrent dataset and saves them directly to Google Drive.

**Requirements:**
- Google Drive with enough storage (5TB recommended)
- The torrent magnet link or `.torrent` file for the PPT Online dataset

**Pipeline:**
1. Mount Google Drive
2. Install `libtorrent`
3. Start torrent download → filter only `.pptx` files
4. Move files to Drive in real-time
5. Track progress with resume support

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CONFIG ──────────────────────────────────────────────────────────────────
DRIVE_SAVE_FOLDER = '/content/drive/MyDrive/PPT_Dataset'   # destination folder
MAX_FILES         = 100_000                                 # stop after this many .pptx files
TEMP_DOWNLOAD_DIR = '/content/torrent_tmp'                  # Colab local temp dir
TORRENT_SAVE_DIR  = '/content/torrent_state'                # resumable state dir
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_SAVE_FOLDER, exist_ok=True)
os.makedirs(TEMP_DOWNLOAD_DIR, exist_ok=True)
os.makedirs(TORRENT_SAVE_DIR,  exist_ok=True)

print(f'✅ Drive mounted. Saving to: {DRIVE_SAVE_FOLDER}')

## Step 2 — Install libtorrent

In [ ]:
!apt-get install -qq python3-libtorrent
!pip install -q tqdm

import libtorrent as lt
print(f'✅ libtorrent version: {lt.version}')

## Step 3 — Configure Torrent

Paste your **magnet link** or provide a path to the `.torrent` file below.

In [ ]:
# ── PASTE YOUR MAGNET LINK HERE ──────────────────────────────────────────────
MAGNET_LINK = 'magnet:?xt=urn:btih:XXXXXXXXXXXXXXXXXXXX&dn=PPT+Online+Dataset'

# OR provide a path to a .torrent file (leave MAGNET_LINK blank if using this)
TORRENT_FILE = ''  # e.g. '/content/drive/MyDrive/pptonline.torrent'
# ─────────────────────────────────────────────────────────────────────────────

if not MAGNET_LINK and not TORRENT_FILE:
    raise ValueError('Set MAGNET_LINK or TORRENT_FILE above.')

print('✅ Torrent source configured.')

## Step 4 — Load Progress Tracker (Resume Support)

In [ ]:
import json, hashlib, shutil, time
from pathlib import Path
from tqdm.notebook import tqdm

PROGRESS_FILE = '/content/drive/MyDrive/PPT_Dataset/.download_progress.json'

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {'downloaded': [], 'count': 0, 'failed': []}

def save_progress(progress):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f, indent=2)

progress = load_progress()
already_downloaded = set(progress['downloaded'])
print(f'📂 Already downloaded: {progress["count"]:,} files — resuming from here.')

## Step 5 — Start Download + Auto-Move to Drive

In [ ]:
import threading

# ── Session setup ────────────────────────────────────────────────────────────
ses = lt.session()
ses.listen_on(6881, 6891)

settings = ses.get_settings()
settings['active_downloads'] = 1
settings['connections_limit'] = 200
ses.apply_settings(settings)

# ── Add torrent ──────────────────────────────────────────────────────────────
params = {
    'save_path': TEMP_DOWNLOAD_DIR,
    'storage_mode': lt.storage_mode_t.storage_mode_sparse,
}

if TORRENT_FILE:
    info = lt.torrent_info(TORRENT_FILE)
    params['ti'] = info
    handle = ses.add_torrent(params)
else:
    handle = lt.add_magnet_uri(ses, MAGNET_LINK, params)
    print('⏳ Fetching torrent metadata from magnet link...')
    while not handle.has_metadata():
        time.sleep(1)
    print('✅ Metadata received!')

torrent_info = handle.get_torrent_info()
all_files    = torrent_info.files()
total_files  = all_files.num_files()

# ── Identify .pptx files only ─────────────────────────────────────────────
pptx_indices = [
    i for i in range(total_files)
    if all_files.file_path(i).lower().endswith('.pptx')
]

print(f'📊 Total files in torrent : {total_files:,}')
print(f'📊 .pptx files found      : {len(pptx_indices):,}')
print(f'🎯 Target to download     : {min(MAX_FILES, len(pptx_indices)):,}')

# ── Prioritise only .pptx files (skip everything else) ───────────────────
file_priorities = [0] * total_files   # 0 = skip
for i in pptx_indices[:MAX_FILES]:
    file_priorities[i] = 1            # 1 = normal priority
handle.prioritize_files(file_priorities)

print('\n🚀 Download started! Files will be moved to Drive as they complete...')

## Step 6 — Monitor, Move Files & Track Progress

In [ ]:
from IPython.display import clear_output

def bytes_to_human(n):
    for unit in ['B','KB','MB','GB','TB']:
        if n < 1024: return f'{n:.1f} {unit}'
        n /= 1024
    return f'{n:.1f} PB'

moved_this_session = 0
CHECK_INTERVAL     = 10   # seconds between checks
SAVE_EVERY         = 50   # save progress JSON every N moves

target_count = min(MAX_FILES, len(pptx_indices))
pbar = tqdm(total=target_count, desc='Moving to Drive', unit='file',
            initial=progress['count'])

try:
    while True:
        s = handle.status()

        # ── Scan for newly completed .pptx files ──────────────────────────
        for idx in pptx_indices[:MAX_FILES]:
            rel_path  = all_files.file_path(idx)
            full_src  = os.path.join(TEMP_DOWNLOAD_DIR, rel_path)
            file_name = os.path.basename(rel_path)

            if file_name in already_downloaded:
                continue
            if not os.path.exists(full_src):
                continue

            # File exists → move to Drive
            dst_path = os.path.join(DRIVE_SAVE_FOLDER, file_name)

            # Handle name collisions
            if os.path.exists(dst_path):
                stem = Path(file_name).stem
                dst_path = os.path.join(DRIVE_SAVE_FOLDER, f'{stem}_{idx}.pptx')

            try:
                shutil.move(full_src, dst_path)
                already_downloaded.add(file_name)
                progress['downloaded'].append(file_name)
                progress['count'] += 1
                moved_this_session += 1
                pbar.update(1)
            except Exception as e:
                progress['failed'].append({'file': file_name, 'error': str(e)})

            # Periodic progress save
            if moved_this_session % SAVE_EVERY == 0 and moved_this_session > 0:
                save_progress(progress)

            # Stop if target reached
            if progress['count'] >= MAX_FILES:
                print(f'\n🎉 Target reached: {MAX_FILES:,} files downloaded!')
                save_progress(progress)
                handle.pause()
                pbar.close()
                break

        else:
            # ── Status display ────────────────────────────────────────────
            clear_output(wait=True)
            print(f'📥 Download speed    : {bytes_to_human(s.download_rate)}/s')
            print(f'📤 Upload speed      : {bytes_to_human(s.upload_rate)}/s')
            print(f'🌐 Peers connected   : {s.num_peers}')
            print(f'📊 Torrent progress  : {s.progress * 100:.2f}%')
            print(f'✅ Moved this session: {moved_this_session:,} files')
            print(f'📂 Total in Drive    : {progress["count"]:,} / {target_count:,}')
            print(f'❌ Failed            : {len(progress["failed"]):,}')
            print(f'\nState: {str(s.state)}')
            time.sleep(CHECK_INTERVAL)
            continue
        break

except KeyboardInterrupt:
    print('\n⏸ Interrupted — saving progress...')
    save_progress(progress)
    print(f'✅ Progress saved. {progress["count"]:,} files in Drive so far.')

## Step 7 — Final Report

In [ ]:
progress = load_progress()

# Count files actually in Drive folder
drive_files = [f for f in os.listdir(DRIVE_SAVE_FOLDER) if f.endswith('.pptx')]

print('=' * 50)
print('📋 DOWNLOAD SUMMARY')
print('=' * 50)
print(f'✅ Files in Drive     : {len(drive_files):,}')
print(f'📝 Progress log count : {progress["count"]:,}')
print(f'❌ Failed transfers   : {len(progress["failed"]):,}')
print(f'📁 Save location      : {DRIVE_SAVE_FOLDER}')
print('=' * 50)

if progress['failed']:
    print('\n⚠️ Failed files (first 10):')
    for f in progress['failed'][:10]:
        print(f'  - {f["file"]}: {f["error"]}')

## (Optional) Step 8 — Retry Failed Files

In [ ]:
# Re-attempt any files that failed to move
retry_list = progress.get('failed', [])
still_failed = []

for item in retry_list:
    fname = item['file']
    # Search for file in temp dir (may have reappeared)
    for root, _, files in os.walk(TEMP_DOWNLOAD_DIR):
        if fname in files:
            src = os.path.join(root, fname)
            dst = os.path.join(DRIVE_SAVE_FOLDER, fname)
            try:
                shutil.copy2(src, dst)
                print(f'✅ Retried: {fname}')
            except Exception as e:
                still_failed.append({'file': fname, 'error': str(e)})
            break
    else:
        still_failed.append(item)

progress['failed'] = still_failed
save_progress(progress)
print(f'\n🔁 Retry done. Still failing: {len(still_failed)}')